# Neo4j Integration with NEUIToolkit

This notebook demonstrates how to export extracted knowledge graphs to Neo4j and query them using Cypher.

## Prerequisites

1. **Neo4j Database**: Running locally or in the cloud
   - Local: `docker run -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:5-community`
   - Cloud: [Neo4j Aura](https://neo4j.com/cloud/aura/)

2. **Configuration**: Set Neo4j credentials in `.env`:
   ```bash
   ENABLE_NEO4J=true
   NEO4J_URI=bolt://localhost:7687
   NEO4J_USERNAME=neo4j
   NEO4J_PASSWORD=password
   ```

## What You'll Learn

- How to connect to Neo4j from NEUIToolkit
- Export knowledge graphs to Neo4j
- Query knowledge using Cypher
- Visualize graph database results
- Best practices for graph database management

## Setup

In [ ]:
import sys
import os
from pathlib import Path
import json

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import NEUIToolkit modules
from backend.neo4j_integration import Neo4jConnector, Neo4jConfig
from backend.orchestrator import run_entity_pass, run_relationship_pass, run_rule_pass
from backend.quality_assurance import QualityAssurance

print("✅ Modules loaded successfully!")

## Step 1: Configure Neo4j Connection

Create a connection configuration using environment variables or explicit parameters.

In [ ]:
# Option 1: Load from environment variables
config = Neo4jConfig(
    uri=os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    username=os.getenv("NEO4J_USERNAME", "neo4j"),
    password=os.getenv("NEO4J_PASSWORD", "password"),
    database=os.getenv("NEO4J_DATABASE", "neo4j")
)

# Option 2: Explicit configuration
# config = Neo4jConfig(
#     uri="bolt://localhost:7687",
#     username="neo4j",
#     password="your_password",
#     database="neo4j"
# )

print(f"Neo4j Configuration:")
print(f"  URI: {config.uri}")
print(f"  Database: {config.database}")
print(f"  Username: {config.username}")

## Step 2: Test Connection

Verify that we can connect to Neo4j.

In [ ]:
try:
    with Neo4jConnector(config) as connector:
        # Test connection by running a simple query
        result = connector.execute_query("RETURN 'Hello from Neo4j!' as message")
        print("✅ Connection successful!")
        print(f"Message from Neo4j: {result[0]['message']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Check that Neo4j is running")
    print("  2. Verify credentials in .env file")
    print("  3. Check firewall settings")
    print("  4. Ensure Neo4j is listening on correct port")

## Step 3: Extract Knowledge from Sample Text

First, let's extract some knowledge to import into Neo4j.

In [ ]:
# Sample text about neuroscience
sample_text = """
The human brain consists of billions of neurons that communicate through synapses.
Neurons use neurotransmitters to send signals across synaptic gaps. The hippocampus
plays a crucial role in memory formation and spatial navigation.

The prefrontal cortex is responsible for executive functions such as decision-making,
planning, and impulse control. The amygdala processes emotions, particularly fear and
threat detection.

If a neuron receives sufficient excitatory signals, then it fires an action potential.
If the hippocampus is damaged, then new memories cannot be formed effectively.
"""

print("Extracting knowledge from sample text...\n")

# Run extraction
entities = run_entity_pass(sample_text)
relationships = run_relationship_pass(entities, sample_text)
rules = run_rule_pass(sample_text)

# Apply quality assurance
qa = QualityAssurance(min_confidence=0.5)
entities, entity_metrics = qa.filter_low_quality(entities, 'entity')
relationships, rel_metrics = qa.filter_low_quality(relationships, 'relationship')
rules, rule_metrics = qa.filter_low_quality(rules, 'rule')

print(f"✅ Extraction complete:")
print(f"   Entities: {len(entities)}")
print(f"   Relationships: {len(relationships)}")
print(f"   Rules: {len(rules)}")
print(f"   Overall Quality: {entity_metrics['quality_score']:.3f}")

## Step 4: Initialize Graph Schema

Before importing data, let's set up the graph schema with constraints and indexes.

In [ ]:
with Neo4jConnector(config) as connector:
    print("Creating graph schema...\n")
    
    # Create schema (constraints and indexes)
    connector.create_schema()
    
    # Verify schema
    schema_info = connector.validate_graph_schema()
    
    print("✅ Schema created!\n")
    print("Current database statistics:")
    print(f"  Nodes: {schema_info.get('node_count', 0)}")
    print(f"  Relationships: {schema_info.get('relationship_count', 0)}")
    print(f"  Constraints: {schema_info.get('constraint_count', 0)}")
    print(f"  Indexes: {schema_info.get('index_count', 0)}")

## Step 5: Import Knowledge Graph

Import the extracted knowledge into Neo4j.

In [ ]:
with Neo4jConnector(config) as connector:
    print("Importing knowledge graph...\n")
    
    # Import all extracted knowledge
    stats = connector.import_knowledge_graph(
        entities=entities,
        relationships=relationships,
        rules=rules,
        document_name="neuroscience_sample.txt"
    )
    
    print("✅ Import complete!\n")
    print("Import statistics:")
    print(f"  Entities created: {stats.get('entities', 0)}")
    print(f"  Relationships created: {stats.get('relationships', 0)}")
    print(f"  Rules created: {stats.get('rules', 0)}")
    print(f"  Documents tracked: {stats.get('documents', 0)}")

## Step 6: Query Knowledge with Cypher

Now let's query the knowledge graph using Cypher.

### Query 1: Find All Entities

In [ ]:
with Neo4jConnector(config) as connector:
    entities = connector.query_knowledge(limit=20)
    
    print(f"Found {len(entities)} entities:\n")
    for entity in entities:
        print(f"  • {entity['name']} ({entity['category']})")
        if entity.get('aliases'):
            print(f"    Aliases: {', '.join(entity['aliases'])}")

### Query 2: Find Entities by Category

In [ ]:
with Neo4jConnector(config) as connector:
    # Find all brain structures
    structures = connector.query_knowledge(category="Structure", limit=10)
    
    print(f"Brain Structures ({len(structures)}):")
    for s in structures:
        print(f"  • {s['name']}")

### Query 3: Explore Relationships

In [ ]:
with Neo4jConnector(config) as connector:
    # Find all relationships involving "Neuron"
    query = """
    MATCH (s:Entity)-[r:RELATES_TO]->(o:Entity)
    WHERE s.name CONTAINS 'Neuron' OR o.name CONTAINS 'Neuron'
    RETURN s.name as subject, r.predicate as relationship, o.name as object,
           r.justification as justification
    LIMIT 10
    """
    
    results = connector.execute_query(query)
    
    print("Relationships involving neurons:\n")
    for i, r in enumerate(results, 1):
        print(f"{i}. {r['subject']} → {r['relationship']} → {r['object']}")
        if r.get('justification'):
            print(f"   Justification: {r['justification']}")
        print()

### Query 4: Find Connection Paths

In [ ]:
with Neo4jConnector(config) as connector:
    # Find shortest path between two entities
    query = """
    MATCH path = shortestPath(
        (start:Entity {name: 'Neuron'})-[:RELATES_TO*1..3]-(end:Entity {name: 'Memory'})
    )
    RETURN [node in nodes(path) | node.name] as path_nodes,
           [rel in relationships(path) | rel.predicate] as path_relationships
    LIMIT 5
    """
    
    paths = connector.execute_query(query)
    
    print("Connection paths from Neuron to Memory:\n")
    for i, path in enumerate(paths, 1):
        nodes = path['path_nodes']
        rels = path['path_relationships']
        
        path_str = nodes[0]
        for j, rel in enumerate(rels):
            path_str += f" → [{rel}] → {nodes[j+1]}"
        
        print(f"{i}. {path_str}")

### Query 5: Analyze Graph Structure

In [ ]:
with Neo4jConnector(config) as connector:
    # Find most connected entities (hubs)
    query = """
    MATCH (e:Entity)-[r:RELATES_TO]-()
    RETURN e.name as entity, e.category as category,
           count(r) as connection_count
    ORDER BY connection_count DESC
    LIMIT 10
    """
    
    hubs = connector.execute_query(query)
    
    print("Most connected entities (hubs):\n")
    for i, hub in enumerate(hubs, 1):
        print(f"{i}. {hub['entity']} ({hub['category']}): {hub['connection_count']} connections")

### Query 6: Retrieve Rules

In [ ]:
with Neo4jConnector(config) as connector:
    # Find all logical rules
    query = """
    MATCH (r:Rule)
    RETURN r.rule_id as id, r.if_clause as if_clause,
           r.then_clause as then_clause, r.confidence as confidence
    ORDER BY r.confidence DESC
    """
    
    rules = connector.execute_query(query)
    
    print(f"Extracted Rules ({len(rules)}):\n")
    for i, rule in enumerate(rules, 1):
        print(f"{i}. IF: {rule['if_clause']}")
        print(f"   THEN: {rule['then_clause']}")
        print(f"   Confidence: {rule.get('confidence', 'N/A')}")
        print()

## Step 7: Visualize Graph (Simple)

Create a simple visualization of the knowledge graph.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

with Neo4jConnector(config) as connector:
    # Get all relationships
    query = """
    MATCH (s:Entity)-[r:RELATES_TO]->(o:Entity)
    RETURN s.name as source, o.name as target, r.predicate as predicate
    LIMIT 50
    """
    
    relationships = connector.execute_query(query)
    
    # Create NetworkX graph
    G = nx.DiGraph()
    
    for rel in relationships:
        G.add_edge(rel['source'], rel['target'], label=rel['predicate'])
    
    # Visualize
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=2, iterations=50)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                          node_size=3000, alpha=0.9)
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, edge_color='gray', 
                          arrows=True, arrowsize=20, width=2)
    
    # Draw labels
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')
    
    # Draw edge labels
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7)
    
    plt.title("Knowledge Graph from Neo4j", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"\nGraph Statistics:")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")
    print(f"  Density: {nx.density(G):.3f}")

## Step 8: Advanced Queries

Let's try some more advanced Cypher patterns.

### Pattern 1: Find Common Neighbors

In [ ]:
with Neo4jConnector(config) as connector:
    # Find entities that share common connections
    query = """
    MATCH (e1:Entity)-[:RELATES_TO]-(common:Entity)-[:RELATES_TO]-(e2:Entity)
    WHERE id(e1) < id(e2)  // Avoid duplicates
    WITH e1.name as entity1, e2.name as entity2, 
         collect(DISTINCT common.name) as common_connections
    WHERE size(common_connections) > 0
    RETURN entity1, entity2, common_connections,
           size(common_connections) as connection_count
    ORDER BY connection_count DESC
    LIMIT 5
    """
    
    results = connector.execute_query(query)
    
    print("Entities with common connections:\n")
    for r in results:
        print(f"{r['entity1']} ↔ {r['entity2']}")
        print(f"  Common connections ({r['connection_count']}): {', '.join(r['common_connections'])}")
        print()

### Pattern 2: Category Distribution

In [ ]:
with Neo4jConnector(config) as connector:
    # Analyze entity distribution by category
    query = """
    MATCH (e:Entity)
    RETURN e.category as category, count(e) as count
    ORDER BY count DESC
    """
    
    distribution = connector.execute_query(query)
    
    print("Entity Category Distribution:\n")
    for cat in distribution:
        print(f"  {cat['category']}: {cat['count']}")
    
    # Visualize
    import matplotlib.pyplot as plt
    
    categories = [d['category'] for d in distribution]
    counts = [d['count'] for d in distribution]
    
    plt.figure(figsize=(10, 6))
    plt.bar(categories, counts, color='skyblue')
    plt.xlabel('Category')
    plt.ylabel('Count')
    plt.title('Entity Distribution by Category')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Step 9: Database Management

### Clear Database (Careful!)

In [ ]:
# CAUTION: This will delete ALL data in the database!
# Uncomment to run

# with Neo4jConnector(config) as connector:
#     # Delete all nodes and relationships
#     connector.execute_query("MATCH (n) DETACH DELETE n")
#     print("✅ Database cleared")

### Export Data from Neo4j

In [ ]:
with Neo4jConnector(config) as connector:
    # Export all entities
    entities = connector.query_knowledge(limit=1000)
    
    # Save to JSON
    output_dir = Path("./neo4j_export")
    output_dir.mkdir(exist_ok=True)
    
    with open(output_dir / "entities.json", "w") as f:
        json.dump(entities, f, indent=2)
    
    print(f"✅ Exported {len(entities)} entities to {output_dir}/entities.json")

## Best Practices

### 1. Connection Management
- Always use context manager (`with`) for automatic cleanup
- Don't keep connections open longer than necessary
- Configure connection pool size based on workload

### 2. Query Optimization
- Use LIMIT to prevent loading too much data
- Create indexes on frequently queried properties
- Use PROFILE to analyze query performance
- Avoid Cartesian products in patterns

### 3. Data Modeling
- Use meaningful relationship types
- Store frequently accessed data as node properties
- Consider denormalization for performance
- Use constraints to ensure data integrity

### 4. Batch Imports
- Use batch inserts for large datasets
- Consider using APOC procedures
- Monitor memory usage during imports

## Next Steps

1. **Process Real Documents**: Import knowledge from your own documents
2. **Complex Queries**: Explore graph algorithms (PageRank, community detection)
3. **Visualization**: Use Neo4j Browser or Bloom for interactive exploration
4. **Integration**: Build applications that query Neo4j
5. **Performance**: Optimize indexes and query patterns

## Resources

- [Neo4j Cypher Manual](https://neo4j.com/docs/cypher-manual/current/)
- [Graph Data Science Library](https://neo4j.com/docs/graph-data-science/current/)
- [NEUIToolkit Documentation](../README.md)
- [APOC Procedures](https://neo4j.com/labs/apoc/)